In [ ]:

from transformers import AutoModelForMultimodalLM
from transformers import AutoProcessor

/home/aash/miniconda3/envs/captioningenv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
from helpers.captioner_utils import *
from helpers.messages import *

In [ ]:
video_path = "/home/aash/Datasets/tvsum/video_1.mp4"

In [2]:
model_name = "HuggingFaceTB/SmolVLM2-2.2B-Instruct"

In [3]:
model = AutoModelForMultimodalLM.from_pretrained( model_name,
device_map="auto" 
)
processor = AutoProcessor.from_pretrained(model_name)
model.eval()


[transformers] Model config: pad_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 128002. This may result in unexpected behavior.
Loading weights: 100%|██████████| 657/657 [00:01<00:00, 649.81it/s]


SmolVLMForConditionalGeneration(
  (model): SmolVLMModel(
    (vision_model): SmolVLMVisionTransformer(
      (embeddings): SmolVLMVisionEmbeddings(
        (patch_embedding): Conv2d(3, 1152, kernel_size=(14, 14), stride=(14, 14), padding=valid)
        (position_embedding): Embedding(729, 1152)
      )
      (encoder): SmolVLMEncoder(
        (layers): ModuleList(
          (0-26): 27 x SmolVLMEncoderLayer(
            (self_attn): SmolVLMVisionAttention(
              (k_proj): Linear(in_features=1152, out_features=1152, bias=True)
              (v_proj): Linear(in_features=1152, out_features=1152, bias=True)
              (q_proj): Linear(in_features=1152, out_features=1152, bias=True)
              (out_proj): Linear(in_features=1152, out_features=1152, bias=True)
            )
            (layer_norm1): LayerNorm((1152,), eps=1e-06, elementwise_affine=True, bias=True)
            (mlp): SmolVLMVisionMLP(
              (activation_fn): GELUTanh()
              (fc1): Linear(in_feat

In [7]:
# Test if captioning works out of the box?
messages = messages = [
    {
        "role": "user",
        "content": [
            {"type": "video", "url": video_path},
            {"type": "text", "text": "Concisely Describe this video"},
        ]
    },
]

In [10]:
inputs = processor.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
).to(model.device, dtype=torch.bfloat16)
input_lens = inputs['input_ids'].shape[-1]
generated_ids = model.generate(**inputs, do_sample=False, max_new_tokens=512)
generated_texts = processor.batch_decode(
    generated_ids,
    skip_special_tokens=True,
)
print(generated_texts[0][input_lens:])

[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`


In [11]:
len(generated_texts[0])

1944

In [13]:
print(generated_texts[0])

User: You are provided the following series of sixty-four frames from a 0:05:53 [H:MM:SS] video.

Frame from 00:00:
Frame from 00:06:
Frame from 00:12:
Frame from 00:17:
Frame from 00:23:
Frame from 00:28:
Frame from 00:34:
Frame from 00:40:
Frame from 00:45:
Frame from 00:51:
Frame from 00:56:
Frame from 01:02:
Frame from 01:07:
Frame from 01:13:
Frame from 01:19:
Frame from 01:24:
Frame from 01:30:
Frame from 01:35:
Frame from 01:41:
Frame from 01:47:
Frame from 01:52:
Frame from 01:58:
Frame from 02:03:
Frame from 02:09:
Frame from 02:14:
Frame from 02:20:
Frame from 02:26:
Frame from 02:31:
Frame from 02:37:
Frame from 02:42:
Frame from 02:48:
Frame from 02:53:
Frame from 02:59:
Frame from 03:05:
Frame from 03:10:
Frame from 03:16:
Frame from 03:21:
Frame from 03:27:
Frame from 03:33:
Frame from 03:38:
Frame from 03:44:
Frame from 03:49:
Frame from 03:55:
Frame from 04:00:
Frame from 04:06:
Frame from 04:12:
Frame from 04:17:
Frame from 04:23:
Frame from 04:28:
Frame from 04:34:
Fr

# Gemma 4 test

In [ ]:
from helpers.captioner_utils import run_video_caption_gemma,run_image_wise_description
from helpers.messages import *
from helpers.video_helpers import encode_batch_image_and_message

In [ ]:
model_name = 'google/gemma-4-E2B-it'
model = AutoModelForMultimodalLM.from_pretrained( # Changed from AutoModelForVision2Seq
        model_name,
    device_map="auto" # Automatically map model to available devices (GPU if present)
)
processor = AutoProcessor.from_pretrained(model_name)
model.eval()

Loading weights: 100%|██████████| 1951/1951 [00:01<00:00, 1203.65it/s]


Gemma4ForConditionalGeneration(
  (model): Gemma4Model(
    (vision_tower): Gemma4VisionModel(
      (patch_embedder): Gemma4VisionPatchEmbedder(
        (input_proj): Linear(in_features=768, out_features=768, bias=False)
      )
      (encoder): Gemma4VisionEncoder(
        (rotary_emb): Gemma4VisionRotaryEmbedding()
        (layers): ModuleList(
          (0-15): 16 x Gemma4VisionEncoderLayer(
            (self_attn): Gemma4VisionAttention(
              (q_proj): Gemma4ClippableLinear(
                (linear): Linear(in_features=768, out_features=768, bias=False)
              )
              (k_proj): Gemma4ClippableLinear(
                (linear): Linear(in_features=768, out_features=768, bias=False)
              )
              (v_proj): Gemma4ClippableLinear(
                (linear): Linear(in_features=768, out_features=768, bias=False)
              )
              (o_proj): Gemma4ClippableLinear(
                (linear): Linear(in_features=768, out_features=768, bias=Fals

In [9]:
import torch
torch.cuda.empty_cache()
import gc
gc.collect()

618

In [ ]:
generated_outputs = run_image_wise_description(video_path,model,processor,independent_frame_message,10,500)

[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[W925 07:44:11.606275447 CUDACachingAllocator.cpp:3933] memory allocation failed with OOM on device 0 while trying to allocate 1631584256 bytes (free: 1424490496, total: 16698769408).
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__c

In [17]:
generated_outputs

[['', '', '', '', '', '', '', '', '', ''],
 ['', '', '', '', '', '', '', '', '', ''],
 ['', '', '', '', '', '', '', '', '', ''],
 ['', '', '', '', '', '', '', '', '', ''],
 ['', '', '', '', '', '', '', '', '', ''],
 ['', '', '', '', '', '', '', '', '', ''],
 ['', '', '', '', '', '', '', '', '', ''],
 ['', '', '', '', '', '', '', '', '', ''],
 ['', '', '', '', '', '', '', '', '', ''],
 ['', '', '', '', '', '', '', '', '', ''],
 ['', '', '', '', '', '', '', '', '', ''],
 ['', '', '', '', '', '', '', '', '', ''],
 ['', '', '', '', '', '', '', '', '', ''],
 ['', '', '', '', '', '', '', '', '', ''],
 ['', '', '', '', '', '', '', '', '', ''],
 ['', '', '', '', '', '', '', '', '', ''],
 ['', '', '', '', '', '', '', '', '', ''],
 ['', '', '', '', '', '', '', '', '', ''],
 ['', '', '', '', '', '', '', '', '', ''],
 ['', '', '', '', '', '', '', '', '', ''],
 ['', '', '', '', '', '', '', '', '', ''],
 ['', '', '', '', '', '', '', '', '', ''],
 ['', '', '', '', '', '', '', '', '', ''],
 ['', '', '

In [5]:
generated_outputs = run_video_caption_gemma(video_path,model,processor,chunked_video_message,9)

[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transform

In [6]:
generated_outputs

'A man with gray hair is looking towards the camera while standing outdoors near a house with a wooden structure and some vegetation.\nThe man is looking directly at the camera with an open mouth and a visible mustache.\nThe ground in the foreground shows a tire, a spray bottle, and a green object, with shadows from tree branches cast on the ground.\nThe man continues to look towards the camera with an open mouth, speaking or reacting to something, while positioned between a wooden fence and a red brick wall.\nA maroon minivan is visible in the background on a gravel driveway.\nThe video shows a maroon minivan parked on a gravel driveway next to a wooden fence. Bright sun glare reflects off the vehicle\'s side and hood.\nThe video now shows a close-up of the front quarter of a maroon minivan parked outdoors on a gravel area next to a wooden fence. Bright sun glare reflects intensely off the side of the vehicle, highlighting the paintwork. The front of the minivan is visible, including 

In [7]:
import gc
import torch
torch.cuda.empty_cache()
gc.collect()

48

In [4]:
from helpers.text_refiners import refine_caption
from gemma_4.captioner_utils import forward_and_decode

In [13]:
refined_message,input_len = refine_caption(processor,generated_outputs)
generated_outputs = forward_and_decode(model, refined_message, input_len,processor,768)

[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`


In [14]:
generated_outputs

'A man with gray hair and a mustache looks directly at the camera with an open mouth, standing outdoors near a house with a wooden structure and vegetation, positioned between a wooden fence and a red brick wall. The ground in the foreground shows a tire, a spray bottle, and a green object, with shadows from tree branches. A maroon minivan is visible in the background on a gravel driveway, with bright sun glare reflecting off it. Close-ups show various interactions with a tire and wheel hub, including hands touching the tread, pointing at star-shaped holes in the hub, gripping objects, and using a metal rod to interact with the hub. One shot shows a silhouette of a person against a bright outdoor setting with a white spray bottle in the middle ground.'

In [ ]:
from transformers import AutoProcessor, AutoModelForMultimodalLM

processor = AutoProcessor.from_pretrained("Qwen/Qwen3.5-4B")
model = AutoModelForMultimodalLM.from_pretrained("Qwen/Qwen3.5-4B", device_map="cuda:1")


/home/aash/miniconda3/envs/captioningenv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 2 files: 100%|██████████| 2/2 [06:42<00:00, 201.33s/it]
[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d
Loading weights: 100%|██████████| 723/723 [00:05<00:00, 124.33it/s]


NameError: name 'video_path' is not defined

In [13]:
messages = [
    {
        "role": "user",
        "content": [
            {"type": "video", "video":video_path },
            {"type": "text", "text": "Generate a concise description of this video. Do not include any time-stamps. Return the summary as one paragraph"}
        ]
    },
]

inputs = processor.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",enable_thinking=False
).to(model.device)



In [ ]:
outputs = model.generate(**inputs, max_new_tokens=1500)


The video showcases a man with a mustache demonstrating how to fix a flat tire on a car. He begins by showing the flat tire and then proceeds to remove it from the vehicle. The man then uses a jack to lift the car and replaces the flat tire with a new one. Throughout the video, he provides tips and advice on how to properly change a tire. The video ends with the man speaking to the camera, likely summarizing the process or offering additional advice.<|im_end|>
<|endoftext|>


In [16]:
print(processor.decode(outputs[0][inputs["input_ids"].shape[-1]:],skip_special_tokens=True))

The video showcases a man with a mustache demonstrating how to fix a flat tire on a car. He begins by showing the flat tire and then proceeds to remove it from the vehicle. The man then uses a jack to lift the car and replaces the flat tire with a new one. Throughout the video, he provides tips and advice on how to properly change a tire. The video ends with the man speaking to the camera, likely summarizing the process or offering additional advice.

